### b.

In [1]:
import numpy as np
import itertools

r = 0.0132
sigma1 = 0.2214
sigma2 = 0.3036
rho = 0.237
T = 0.25
S1 = 158.12
S2 = 170.33

def price(tau, s1, s2, v1, v2, corr):
    return np.exp(r*tau)*(s1**2*np.exp(v1**2*tau)
                          - 2*s1*s2*np.exp(v1*v2*corr*tau)
                          + s2**2*np.exp(v2**2*tau))

option = price(T, S1, S2, sigma1, sigma2, rho)
hedge1 = 2*np.exp(r*T)*(S1*np.exp(sigma1**2*T) - S2*np.exp(sigma1*sigma2*rho*T))
hedge2 = 2*np.exp(r*T)*(S2*np.exp(sigma2**2*T) - S1*np.exp(sigma1*sigma2*rho*T))
start = option - hedge1*S1 - hedge2*S2
print("option value", round(option, 4))
print("shares held", round(hedge1, 4), round(hedge2, 4))
print("portfolio value", round(start, 4))

option value 921.7399
shares held -21.9525 31.2019
portfolio value -921.7399


In [2]:
moves = [0.20, 0.10, 0.05, -0.05, -0.10, -0.20]
ratios = [0.5, 0.75, 1.25, 1.5, 1.75, 2.0]
shifts = [0.0, 0.5, -0.5]
grid = list(itertools.product(moves, moves, ratios, ratios, shifts))
tau = T - 5/252

losses = []
for x1, x2, k1, k2, d in grid:
    s1 = S1*np.exp(x1)
    s2 = S2*np.exp(x2)
    later = price(tau, s1, s2, k1*sigma1, k2*sigma2, rho + d) - hedge1*s1 - hedge2*s2
    losses.append(-(later - start))

losses = np.array(losses)
pick = losses.argmax()
x1, x2, k1, k2, d = grid[pick]
print("worst case loss", round(losses[pick], 4))
print("log returns", x1, x2)
print("volatility ratios", k1, k2)
print("correlation shift", d)

worst case loss 1000.3617
log returns 0.2 0.2
volatility ratios 0.5 0.5
correlation shift 0.5


### c.

In [3]:
move_weight = {0.20: 0.5, -0.20: 0.5, 0.10: 0.75, -0.10: 0.75, 0.05: 1.0, -0.05: 1.0}
ratio_weight = {0.5: 0.5, 2.0: 0.5, 0.75: 1.25, 1.75: 1.25, 1.25: 0.75, 1.5: 0.75}
shift_weight = {0.0: 0.5, 0.5: 1.25, -0.5: 1.25}

weights = []
for x1, x2, k1, k2, d in grid:
    weights.append(move_weight[x1]*move_weight[x2]*ratio_weight[k1]*ratio_weight[k2]*shift_weight[d])

weighted = np.array(weights)*losses
pick = weighted.argmax()
x1, x2, k1, k2, d = grid[pick]
print("weighted measure", round(weighted[pick], 4))
print("loss at that scenario", round(losses[pick], 4))
print("log returns", x1, x2)
print("volatility ratios", k1, k2)
print("correlation shift", d)

weighted measure 1313.7859
loss at that scenario 672.6584
log returns 0.05 0.05
volatility ratios 0.75 0.75
correlation shift 0.5
